# CARLA Sensor Logging

experiment for spawning an ego vehicle, attaching RGB/LiDAR/GNSS sensors, and saving a short CSV log while autopilot drives.


In [ ]:
import carla
import random
import time
import numpy as np
import cv2
from datetime import datetime

# connect to the carla server
client = carla.Client('localhost', 2000)
client.set_timeout(10.0)

# get the current world
world = client.get_world()
blueprint_library = world.get_blueprint_library()

print(f"Current map: {world.get_map().name}")


## Spawn Vehicle


In [ ]:
# pick the tesla model 3 blueprint
vehicle_bp = blueprint_library.filter('model3')[0]

# choose a random spawn point
spawn_points = world.get_map().get_spawn_points()
spawn_point = random.choice(spawn_points)

# spawn the vehicle
vehicle = world.spawn_actor(vehicle_bp, spawn_point)

# enable autopilot
vehicle.set_autopilot(True)

# keep actors for cleanup
actors_list = [vehicle]


In [ ]:
# place the spectator behind the car
spectator = world.get_spectator()
transform = vehicle.get_transform()

# camera view behind the car
spectator_location = transform.location + carla.Location(z=3.0) - transform.get_forward_vector() * 5.0
spectator.set_transform(carla.Transform(spectator_location, transform.rotation))


In [ ]:
# latest sensor values
sensor_data = {
    'camera_image': None,
    'lidar_points': 0,
    'min_distance': float('inf'),
    'gps_lat': 0.0,
    'gps_lon': 0.0,
    'gps_alt': 0.0
}


## Camera


In [ ]:
# camera callback
def camera_callback(image):
    # convert the image to a numpy array
    array = np.frombuffer(image.raw_data, dtype=np.uint8)
    array = np.reshape(array, (image.height, image.width, 4))  # BGRA
    array = array[:, :, :3]  # Remove alpha channel, keep BGR
    # switch to rgb
    array = array[:, :, ::-1]

    sensor_data['camera_image'] = array

# camera blueprint
camera_bp = blueprint_library.find('sensor.camera.rgb')
camera_bp.set_attribute('image_size_x', '800')
camera_bp.set_attribute('image_size_y', '600')
camera_bp.set_attribute('fov', '90')

# front camera mount
camera_transform = carla.Transform(carla.Location(x=1.5, z=2.4))

# attach sensor to the vehicle
camera = world.spawn_actor(camera_bp, camera_transform, attach_to=vehicle)
camera.listen(camera_callback)
actors_list.append(camera)


## LiDAR


In [ ]:
import math

# lidar callback
def lidar_callback(point_cloud):
    # count lidar points
    sensor_data['lidar_points'] = len(point_cloud)
    # estimate nearest lidar point
    if len(point_cloud) > 0:
        distances = [math.sqrt(p.point.x**2 + p.point.y**2 + p.point.z**2) for p in point_cloud]
        sensor_data['min_distance'] = min(distances)
    else:
        sensor_data['min_distance'] = float('inf')

# lidar blueprint
lidar_bp = blueprint_library.find('sensor.lidar.ray_cast')
lidar_bp.set_attribute('channels', '32')  # 32 vertical channels
lidar_bp.set_attribute('range', '50')     # 50 meters max range
lidar_bp.set_attribute('points_per_second', '56000')
lidar_bp.set_attribute('rotation_frequency', '10')  # 10 Hz

# roof lidar mount
lidar_transform = carla.Transform(carla.Location(x=0.0, z=2.5))

# attach sensor to the vehicle
lidar = world.spawn_actor(lidar_bp, lidar_transform, attach_to=vehicle)
lidar.listen(lidar_callback)
actors_list.append(lidar)


## GNSS


In [ ]:
# gnss callback
def gps_callback(gps_data):
    sensor_data['gps_lat'] = gps_data.latitude
    sensor_data['gps_lon'] = gps_data.longitude
    sensor_data['gps_alt'] = gps_data.altitude

# gnss blueprint
gps_bp = blueprint_library.find('sensor.other.gnss')

# center mount
gps_transform = carla.Transform(carla.Location(x=0.0, z=0.0))

# attach sensor to the vehicle
gps = world.spawn_actor(gps_bp, gps_transform, attach_to=vehicle)
gps.listen(gps_callback)
actors_list.append(gps)


## Data Log


In [ ]:
import csv

print("collecting data for 20 seconds...\n")

time.sleep(2)

csv_filename = f"sensor_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
csv_file = open(csv_filename, mode='w', newline='')
csv_writer = csv.writer(csv_file)
csv_writer.writerow(['Time_s', 'Speed_kmh', 'GPS_Lat', 'GPS_Lon', 'GPS_Alt', 'Lidar_Points', 'Min_Distance_m'])

spectator = world.get_spectator()
start_time = time.time()
last_log_time = start_time
duration = 20

try:
    while time.time() - start_time < duration:
        world.wait_for_tick() 
        
        current_time = time.time()
        
        # keep spectator near the car
        transform = vehicle.get_transform()
        spectator_location = transform.location + carla.Location(z=3.0) - transform.get_forward_vector() * 5.0
        spectator.set_transform(carla.Transform(spectator_location, transform.rotation))

        # log every 0.5 seconds
        if current_time - last_log_time >= 0.5:
            elapsed = current_time - start_time
            
            # read speed
            velocity = vehicle.get_velocity()
            speed_kmh = 3.6 * np.sqrt(velocity.x**2 + velocity.y**2 + velocity.z**2)
            
            # format lidar distance
            min_dist = sensor_data['min_distance']
            min_str = f"{min_dist:.2f}m" if min_dist != float('inf') else "N/A"
            
            print(f"[{elapsed:5.1f}s] Speed: {speed_kmh:4.1f} km/h | "
                  f"GPS: ({sensor_data['gps_lat']:.6f}, {sensor_data['gps_lon']:.6f}, {sensor_data['gps_alt']:.1f}m) | "
                  f"Lidar: {sensor_data['lidar_points']:5d} pts (Min: {min_str})")
            
            csv_writer.writerow([
                round(elapsed, 1), 
                round(speed_kmh, 2), 
                sensor_data['gps_lat'], 
                sensor_data['gps_lon'], 
                sensor_data['gps_alt'],
                sensor_data['lidar_points'],
                round(min_dist, 2) if min_dist != float('inf') else -1
            ])
            
            last_log_time = current_time
        
except KeyboardInterrupt:
    print("stopped")
finally:
    csv_file.close()
    print(f"saved to {csv_filename}")


## Last Camera Frame


In [ ]:
import matplotlib.pyplot as plt

if sensor_data['camera_image'] is not None:
    plt.figure(figsize=(12, 6))
    plt.imshow(sensor_data['camera_image'])
    plt.title('Camera View from Vehicle')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"Camera resolution: {sensor_data['camera_image'].shape[1]}x{sensor_data['camera_image'].shape[0]}")
else:
    print("No camera image available")


In [ ]:
# cleanup

for actor in actors_list:
    if actor is not None and actor.is_alive:
        if actor.type_id.startswith('sensor'):
            actor.stop()


for actor in reversed(actors_list):
    if actor is not None and actor.is_alive:
        actor.destroy()


actors_list.clear()
